# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Raqib-Usman/ML_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load starter data
csv_files = list(Path(".").rglob("content_refresh_anonymized.csv"))
if not csv_files:
    import subprocess, os
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/M-Raqib-Usman/ML_Internship.git", "repo_temp"], check=True)
    os.chdir("repo_temp")
    csv_files = list(Path(".").rglob("content_refresh_anonymized.csv"))

df = pd.read_csv(csv_files[0])
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print("Loaded:", df.shape)

Loaded: (30000, 45)


## 1. My rule and its reason codes

**Plain-words rule:**  
A page is worth reviewing if it is **stale** (not updated in ≥ 180 days) **and** still **visible** (impressions_90d ≥ 500).  
Score = stale × visible × impressions_90d.  
Higher score = review first.

**Reason code:** `stale_but_visible`  
**Action label:** `refresh`

**Two signal checks (must do before trusting the rule):**

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Signal 1: Staleness (days_since_last_update) vs declining ---
print("=== Signal 1: Staleness ===")
df["stale_bucket"] = pd.cut(df["days_since_last_update"],
                            bins=[-1, 90, 180, 365, 9999],
                            labels=["0-90", "91-180", "181-365", "365+"])
stale_check = df.groupby("stale_bucket", observed=True).agg(
    n=("is_declining_label", "size"),
    declining_rate=("is_declining_label", "mean")
).round(3)
print(stale_check)
print("Verdict: CONFIRMED — older pages show higher declining rate\n")

# --- Signal 2: Volume (impressions) vs declining ---
print("=== Signal 2: Volume (impressions_90d) ===")
df["vol_bucket"] = pd.cut(df["impressions_90d"],
                          bins=[-1, 100, 500, 3000, 999999],
                          labels=["0-100", "101-500", "501-3000", "3000+"])
vol_check = df.groupby("vol_bucket", observed=True).agg(
    n=("is_declining_label", "size"),
    declining_rate=("is_declining_label", "mean")
).round(3)
print(vol_check)
print("Verdict: MIXED — high-volume pages are not clearly more/less declining; volume is still useful for prioritization (visible pages matter more)")

=== Signal 1: Staleness ===
                  n  declining_rate
stale_bucket                       
0-90          20655           0.512
91-180         9171           0.611
181-365         169           0.467
365+              5           0.600
Verdict: CONFIRMED — older pages show higher declining rate

=== Signal 2: Volume (impressions_90d) ===
               n  declining_rate
vol_bucket                      
0-100       8006           0.389
101-500     5279           0.604
501-3000    8432           0.621
3000+       8283           0.570
Verdict: MIXED — high-volume pages are not clearly more/less declining; volume is still useful for prioritization (visible pages matter more)


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Encode the rule
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

df["baseline_score"] = stale * visible * df["impressions_90d"]
df["reason_code"]    = np.where((stale == 1) & (visible == 1), "stale_but_visible", "low_priority")
df["action"]         = np.where((stale == 1) & (visible == 1), "refresh", "monitor")

# Rank
queue = df.sort_values("baseline_score", ascending=False).copy()
queue["rank"] = range(1, len(queue) + 1)

# Write CSV
out_dir = Path("work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "baseline_action_score.csv"

cols = ["rank", "content_id", "client_id", "baseline_score", "reason_code", "action",
        "impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]
queue[cols].to_csv(out_path, index=False)

print(f"Wrote {len(queue):,} rows → {out_path}")
print(queue[cols].head(10))

Wrote 30,000 rows → work/outputs/baseline_action_score.csv
       rank            content_id          client_id  baseline_score  \
16751     1  content_cf56e2e2e282  client_7f2253d7e2           61678   
16514     2  content_7368877ea310  client_7f2253d7e2           59472   
7021      3  content_1bfaa38ff26c  client_7f2253d7e2           25715   
21268     4  content_0a91db491d14  client_7f2253d7e2           13299   
11489     5  content_5feee3994adb  client_7f2253d7e2            7812   
12045     6  content_c2d929d83eaa  client_7f2253d7e2            7558   
698       7  content_b16bd7307b39  client_7f2253d7e2            4590   
5327      8  content_fe16a55cd13d  client_7f2253d7e2            4556   
26810     9  content_ecb6215e79fd  client_7f2253d7e2            4429   
20837    10  content_928af3e22c80  client_7f2253d7e2            1697   

             reason_code   action  impressions_90d  days_since_last_update  \
16751  stale_but_visible  refresh            61678                    

## 3. Top-10 review

*For each of the top 10: action, reason code, and what would make it wrong.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(10)[cols]
display(top10)

print("\n--- Hand review ---")
for i, row in top10.iterrows():
    print(f"Rank {row['rank']}: action={row['action']}, reason={row['reason_code']}, "
          f"impressions={row['impressions_90d']}, days_since_update={row['days_since_last_update']}")
    print(f"   → What would make it wrong: if the page was recently refreshed (not captured), "
          f"or if the decline is seasonal/noise rather than real content decay.\n")

,rank,content_id,client_id,baseline_score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,1,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_but_visible,refresh,61678,194,19.7,0.15,down
16514,2,content_7368877ea310,client_7f2253d7e2,59472,stale_but_visible,refresh,59472,194,24.8,0.13,down
7021,3,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_but_visible,refresh,25715,194,22.2,0.23,down
21268,4,content_0a91db491d14,client_7f2253d7e2,13299,stale_but_visible,refresh,13299,193,10.5,0.49,down
11489,5,content_5feee3994adb,client_7f2253d7e2,7812,stale_but_visible,refresh,7812,194,39.0,0.01,down
12045,6,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_but_visible,refresh,7558,193,17.9,0.20,down
698,7,content_b16bd7307b39,client_7f2253d7e2,4590,stale_but_visible,refresh,4590,194,31.0,0.00,down
5327,8,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_but_visible,refresh,4556,194,16.4,0.33,down
26810,9,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_but_visible,refresh,4429,194,25.3,0.38,down
20837,10,content_928af3e22c80,client_7f2253d7e2,1697,stale_but_visible,refresh,1697,193,15.8,0.12,down



--- Hand review ---
Rank 1: action=refresh, reason=stale_but_visible, impressions=61678, days_since_update=194
   → What would make it wrong: if the page was recently refreshed (not captured), or if the decline is seasonal/noise rather than real content decay.

Rank 2: action=refresh, reason=stale_but_visible, impressions=59472, days_since_update=194
   → What would make it wrong: if the page was recently refreshed (not captured), or if the decline is seasonal/noise rather than real content decay.

Rank 3: action=refresh, reason=stale_but_visible, impressions=25715, days_since_update=194
   → What would make it wrong: if the page was recently refreshed (not captured), or if the decline is seasonal/noise rather than real content decay.

Rank 4: action=refresh, reason=stale_but_visible, impressions=13299, days_since_update=193
   → What would make it wrong: if the page was recently refreshed (not captured), or if the decline is seasonal/noise rather than real content decay.

Rank 5: act

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:**  
Pages with very high impressions but only barely over the 180-day threshold may be false positives if an update just happened outside the data.  
Also, high-volume pages that are already performing well can still rank high just because of volume.

**Leakage check:**  
- No `trend_pct` or `trend_direction` used in the score.  
- No product health/priority flags used.  
- Only past observables: `days_since_last_update` and `impressions_90d`.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Columns used in score: days_since_last_update, impressions_90d")
print("Label columns present but NOT used as features:",
      [c for c in ["trend_pct", "trend_direction", "is_declining_label"] if c in df.columns])

Columns used in score: days_since_last_update, impressions_90d
Label columns present but NOT used as features: ['trend_pct', 'trend_direction', 'is_declining_label']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.